In [2]:
!nvidia-smi

Sun Jan 25 14:17:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.14              Driver Version: 550.54.14      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:17:00.0 Off |                    0 |
| N/A   34C    P0             65W /  300W |       3MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import sys

sys.path.append("..")

In [4]:
import os


# Force video decoding to torchvision to avoid torchcodec runtime issues
os.environ["VIDEO_DECODING_BACKEND"] = "torchvision"
os.environ["TRANSFORMERS_VIDEO_BACKEND"] = "torchvision"

In [5]:
from pathlib import Path

import torch
from transformers import pipeline

In [6]:
model_name = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
use_kv_cache = False
attention_implementation = None
batch_size = 1
num_workers = 0

In [7]:
pipe = pipeline(
    "image-text-to-text",
    model=model_name,
    device_map="auto",
    dtype=torch.bfloat16,
    use_cache=use_kv_cache,
    model_kwargs={"_attn_implementation": attention_implementation}
    if attention_implementation
    else {},
    batch_size=batch_size,
    num_workers=num_workers,
)

# Ensure torchvision backend is used even if torchcodec is installed
if hasattr(pipe, "video_processor"):
    pipe.video_processor.backend = "torchvision"
elif hasattr(pipe, "processor") and hasattr(pipe.processor, "video_processor"):
    pipe.processor.video_processor.backend = "torchvision"


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [8]:
video_folder = Path("../data/inputs/videos")
audio_folder = Path("../data/inputs/audios")
audio_transcript_folder = Path("../data/inputs/audio_transcripts")

num_video_samples = 10
max_frames = 64

In [9]:
import itertools

from src.audio_utils import get_audio_transcript
from src.inference_utils import create_messages

video_id_iter = video_folder.glob("*.mp4")
if num_video_samples > 0:
    video_id_iter = itertools.islice(video_id_iter, num_video_samples)

video_ids = tuple(video_id_iter)
print(f"🚀 Processing {len(video_ids)} videos...")

summary_messages: list[list[dict[str, str | list[dict[str, str]]]]] = []
category_messages: list[list[dict[str, str | list[dict[str, str]]]]] = []
for video_path in video_ids:
    video_id = video_path.stem

    transcript = get_audio_transcript(
        video_id,
        video_path,
        audio_folder,
        audio_transcript_folder,
    )

    # ---- Summary Prompts ----
    summary_msg = create_messages(video_path, transcript, mode="summary")
    summary_messages.append(summary_msg)

    # ---- Category Prompts ----
    category_msg = create_messages(video_path, transcript, mode="category")
    category_messages.append(category_msg)

🚀 Processing 10 videos...


In [10]:
summary_messages

[[{'role': 'user',
   'content': [{'type': 'video',
     'path': '../data/inputs/videos/7349900395843472672.mp4'},
    {'type': 'text',
     'text': "Describe this video in detail. Use the audio transcript to get more context. Audio Transcript: Nur leider wird über diese Wahl hier in der Meinungsdiktatur Deutschland total einseitig berichtet. Eigentlich gibt es nur noch einen objektiven Reporter. Dobriden! Ich bin's, der verbotene Untergrundsender Russia Today. Sie sind für die Spezialoperation in der Ukraine? Ich bin für die Spezialoperation. Wären Sie dann nicht an der Front besser aufgehoben als hier im befaulichen Bund? Ich sag jetzt, ich hab meinen Militärdienst an der Roten Armee schon gemacht. Seiner Zeit, als noch vor Deutschland in Russland gelebt habe. Ich bin jetzt schon 55 und das ist ja viel zu spät. Ich werde sowieso nicht eingezogen. Aber ich habe gegen Tschetschenien gekämpft, seinerzeit, in den 90ern. Also Sie haben Erfahrung mit völkerrechtswidrigen Angriffen. Ja, ich

In [11]:
import time
from typing import Literal

from tqdm import tqdm
from transformers import ImageTextToTextPipeline


def run_inference(
    pipe: ImageTextToTextPipeline,
    messages: list[list[dict]],
    max_frames: int,
    max_new_tokens: int = 140,
    mode: Literal["category", "summary"] = "category",
) -> tuple[list[str], float]:
    """Run inference and measure time."""
    start = time.perf_counter()
    if mode == "category":
        max_new_tokens = 10

    output = pipe(
        text=messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        max_frames=max_frames,
        video_backend="torchvision",
    )  # type: ignore[reportCallIssue]
    text = [
        out[0]["generated_text"][-1]["content"].strip()
        for out in tqdm(output, desc="Running Inference", total=len(messages))
    ]

    return text, time.perf_counter() - start


In [12]:
print("🚀 Running summary inference...")
summaries, summary_time = run_inference(
    pipe,
    summary_messages,
    max_frames=max_frames,
    mode="summary",
)

🚀 Running summary inference...


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8. On Windows, ensure you've installed
             the "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.1+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8: Could not load this library: /AIML/tinyllms/work/ETLLM/VLM-Inference/.venv/lib/python3.12/site-packages/torchcodec/libtorchcodec_core8.so
FFmpeg version 7: Could not load this library: /AIML/tinyllms/work/ETLLM/VLM-Inference/.venv/lib/python3.12/site-packag